[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-1-python-for-ai/04-the-data-stack/code/numpy_pandas.ipynb)

# Class 1.4: The data stack

The slides carry the ideas. Here you run and tweak the code. The Pandas half works a real dataset (`worldometer_coronavirus_summary_data.csv`), which sits next to this notebook in the `code/` folder.

**What we will cover**

- NumPy: arrays and `dtype`, shape and reshape, indexing and slicing
- Boolean masking, broadcasting, axis aggregations, and the dot product
- Pandas: read a real CSV, inspect, select with `.loc` / `.iloc`
- Filter, derive columns, handle missing data, and `groupby`
- Build: a small filter, clean, sort, group pipeline on the dataset

Run each cell, change a value, and run it again.

## NumPy: why arrays

A NumPy array holds one fixed type in a tight block, so whole-array math runs fast and reads short.

In [1]:
import numpy as np

temps_list = [18, 21, 19, 24]     # a plain Python list
temps = np.array(temps_list)      # a NumPy array, one dtype
print(temps.dtype)                # int64
print(temps + 2)                  # every reading, warmed by 2

int64
[20 23 21 26]


## Creating arrays and dtype

In [2]:
print(np.arange(1, 7))            # [1 2 3 4 5 6]
print(np.linspace(0, 10, 5))      # [ 0.   2.5  5.   7.5 10. ]
print(np.zeros(4))                # [0. 0. 0. 0.]
print(np.ones((2, 2)))            # 2x2 of ones
print(np.array([1, 2, 3]).dtype, np.array([1, 2, 3.0]).dtype)  # int64 float64

[1 2 3 4 5 6]
[ 0.   2.5  5.   7.5 10. ]
[0. 0. 0. 0.]
[[1. 1.]
 [1. 1.]]
int64 float64


In [3]:
print(np.linspace(0, 10, 3))

[ 0.  5. 10.]


In [4]:
print(type(np.linspace(0, 10, 5)))

<class 'numpy.ndarray'>


## Shape and reshape

In [5]:
b = np.arange(12)
print(b.shape, b.ndim)            # (12,) 1

grid = b.reshape(3, 4)            # 3 rows, 4 columns
print(grid)
print(grid.shape, grid.ndim)      # (3, 4) 2

print(b.reshape(-1, 6).shape)     # (2, 6)  -1 is inferred

(12,) 1
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(3, 4) 2
(2, 6)


## Indexing and slicing (2D)

`grid` is the 3x4 array from above. Index it with `[row, column]`.

In [6]:
print(grid[0, 3])     # 3        row 0, column 3
print(grid[1])        # [4 5 6 7]  the whole second row
print(grid[:, 0])     # [0 4 8]    the whole first column
print(grid[0:2, 1:3]) # top-left block, rows 0-1 and cols 1-2

3
[4 5 6 7]
[0 4 8]
[[1 2]
 [5 6]]


## Boolean masking

In [7]:
v = np.array([1, 4, 8, 7, 5, 10, 11])

sub_v = v[[True, False, False, True, True, True, False]]  # [1 8 5 10]

print(sub_v)

[ 1  7  5 10]


In [8]:
v[v % 2 == 0]

array([ 4,  8, 10])

In [9]:
ages = np.array([15, 22, 34, 8, 40])

print(ages[ages >= 18])       # [22 34 40]  keep adults
print((ages >= 18).sum())     # 3           how many adults

ages[ages < 18] = 0           # zero out the minors through a mask
print(ages)                   # [ 0 22 34  0 40]

[22 34 40]
3
[ 0 22 34  0 40]


## Vectorized math and broadcasting

Here a per-product `price` vector broadcasts across every store's row.

In [10]:
units = np.array([[2, 1, 3],     # store A: product 2, 1, 3
                  [0, 4, 1]])    # store B

price = np.array([[10], [20]])    # price per product

revenue = units * price          # price stretched across each row
print(revenue)
print("total:", revenue.sum())

[[20 10 30]
 [ 0 80 20]]
total: 160


## Aggregations and the axis argument

In [11]:
rain = np.array([[2, 0, 5],      # week 1: mon, tue, wed
                 [1, 3, 0]])     # week 2

print(rain.sum())            # 11        everything
print(rain.sum(axis=0))      # [3 3 5]   per day, across weeks
print(rain.mean(axis=1))     # [2.33 1.33]  per week, across days

11
[3 3 5]
[2.33333333 1.33333333]


## Vectors and the dot product

A vector stands for something as numbers; the dot product collapses two vectors into one number that grows as they align.

In [12]:
u = np.array([2, 1, 0, 3])
v = np.array([1, 0, 4, 1])

print(np.dot(u, v))    # 2*1 + 1*0 + 0*4 + 3*1 = 5
print(u @ v)           # same, the @ operator

# normalize u to length 1 (a unit vector)
length = np.sqrt((u ** 2).sum())
print("length:", round(length, 3))
print("unit:", (u / length).round(3))

5
5
length: 3.742
unit: [0.535 0.267 0.    0.802]


------

## Pandas: read a real CSV

`read_csv` loads a file into a DataFrame. This one has 226 countries and 12 columns of pandemic summary figures. The file sits beside this notebook in the `code/` folder.

In [13]:
import pandas as pd

covid = pd.read_csv("worldometer_coronavirus_summary_data.csv")
print("shape:", covid.shape)                       # (226, 12)

shape: (226, 12)


In [14]:
covid.head(10)

,country,continent,total_confirmed,total_deaths,total_recovered,active_cases,serious_or_critical,total_cases_per_1m_population,total_deaths_per_1m_population,total_tests,total_tests_per_1m_population,population
0,Afghanistan,Asia,179267,7690.0,162202.0,9375.0,1124.0,4420,190.0,951337.0,23455.0,40560636
1,Albania,Europe,275574,3497.0,271826.0,251.0,2.0,95954,1218.0,1817530.0,632857.0,2871945
2,Algeria,Africa,265816,6875.0,178371.0,80570.0,6.0,5865,152.0,230861.0,5093.0,45325517
3,Andorra,Europe,42156,153.0,41021.0,982.0,14.0,543983,1974.0,249838.0,3223924.0,77495
4,Angola,Africa,99194,1900.0,97149.0,145.0,NaN,2853,55.0,1499795.0,43136.0,34769277
5,Anguilla,North America,2984,9.0,2916.0,59.0,4.0,195646,590.0,51382.0,3368870.0,15252
6,Antigua And Barbuda,North America,7721,137.0,7511.0,73.0,1.0,77646,1378.0,18901.0,190076.0,99439
7,Argentina,South America,9101319,128729.0,8895999.0,76591.0,372.0,197992,2800.0,35716069.0,776974.0,45968174
8,Armenia,Asia,422896,8623.0,412048.0,2225.0,NaN,142219,2900.0,3068217.0,1031834.0,2973558
9,Aruba,North America,35693,213.0,35199.0,281.0,NaN,331689,1979.0,177885.0,1653053.0,107610


In [15]:
covid.tail(4)

,country,continent,total_confirmed,total_deaths,total_recovered,active_cases,serious_or_critical,total_cases_per_1m_population,total_deaths_per_1m_population,total_tests,total_tests_per_1m_population,population
222,Western Sahara,Africa,10,1.0,9.0,0.0,NaN,16,2.0,NaN,NaN,624681
223,Yemen,Asia,11819,2149.0,9009.0,661.0,23.0,381,69.0,265253.0,8543.0,31049015
224,Zambia,Africa,320591,3983.0,315997.0,611.0,NaN,16575,206.0,3452554.0,178497.0,19342381
225,Zimbabwe,Africa,249206,5482.0,242417.0,1307.0,12.0,16324,359.0,2287793.0,149863.0,15265849


In [16]:
type(covid)

pandas.DataFrame

In [17]:
type(covid['country'])

pandas.Series

In [18]:
print(covid[["country", "continent", "total_confirmed", "total_deaths", "population"]].head())

       country continent  total_confirmed  total_deaths  population
0  Afghanistan      Asia           179267        7690.0    40560636
1      Albania    Europe           275574        3497.0     2871945
2      Algeria    Africa           265816        6875.0    45325517
3      Andorra    Europe            42156         153.0       77495
4       Angola    Africa            99194        1900.0    34769277


## Build a DataFrame yourself

`read_csv` is not the only way in. For small or generated data you build a DataFrame directly, two common shapes: a dict of columns, or a list of rows plus column names.

In [19]:
# 1) from a dict: each key is a column name, each value is that column
from_dict = pd.DataFrame({
    "city": ["Cairo", "Oslo", "Lima"],
    "temp_c": [34, 12, 19],
})
print(from_dict)
print()

# 2) from a list of lists: each inner list is a row; name the columns yourself
rows = [
    ["Cairo", 34],
    ["Oslo", 12],
    ["Lima", 19],
]
from_rows = pd.DataFrame(rows, columns=["city", "temp_c"])
print(from_rows)
print()

print("same table, two ways:", from_dict.equals(from_rows))   # True

    city  temp_c
0  Cairo      34
1   Oslo      12
2   Lima      19

    city  temp_c
0  Cairo      34
1   Oslo      12
2   Lima      19

same table, two ways: True


## Inspect before you analyze

`dtypes` shows datatypes of the columns; `describe` summarizes the numbers; `info` (try it) reveals the non-null counts that hint at missing data.

In [20]:
print(covid.dtypes)
print()
print(covid[["total_confirmed", "total_deaths", "population"]].describe())

country                               str
continent                             str
total_confirmed                     int64
total_deaths                      float64
total_recovered                   float64
active_cases                      float64
serious_or_critical               float64
total_cases_per_1m_population       int64
total_deaths_per_1m_population    float64
total_tests                       float64
total_tests_per_1m_population     float64
population                          int64
dtype: object

       total_confirmed  total_deaths    population
count     2.260000e+02  2.180000e+02  2.260000e+02
mean      2.305651e+06  2.884442e+04  3.495521e+07
std       7.575510e+06  9.971254e+04  1.390338e+08
min       2.000000e+00  1.000000e+00  8.050000e+02
25%       2.412600e+04  2.370000e+02  5.605125e+05
50%       1.793750e+05  2.251500e+03  5.800570e+06
75%       1.090902e+06  1.400650e+04  2.187284e+07
max       8.420947e+07  1.026646e+06  1.439324e+09


In [21]:
covid.info()

<class 'pandas.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         226 non-null    str    
 1   continent                       226 non-null    str    
 2   total_confirmed                 226 non-null    int64  
 3   total_deaths                    218 non-null    float64
 4   total_recovered                 204 non-null    float64
 5   active_cases                    204 non-null    float64
 6   serious_or_critical             145 non-null    float64
 7   total_cases_per_1m_population   226 non-null    int64  
 8   total_deaths_per_1m_population  218 non-null    float64
 9   total_tests                     212 non-null    float64
 10  total_tests_per_1m_population   212 non-null    float64
 11  population                      226 non-null    int64  
dtypes: float64(7), int64(3), str(2)
memory usage: 2

## Indexing with .loc and .iloc

Rows are sorted alphabetically by country, with a default integer index (0 is Afghanistan).

In [22]:
print(covid[["country", "total_confirmed"]].head(3))   # a two-column DataFrame
print()
print("by label   :", covid.loc[0, "country"])         # row-label 0
print("by position:")
print(covid.iloc[0:3, [0, 1]])                          # first 3 rows, country + continent

       country  total_confirmed
0  Afghanistan           179267
1      Albania           275574
2      Algeria           265816

by label   : Afghanistan
by position:
       country continent
0  Afghanistan      Asia
1      Albania    Europe
2      Algeria    Africa


## Filter rows by condition

In [23]:
# countries with more than 20 million confirmed cases
big = covid[covid["total_confirmed"] > 20_000_000]
print(big[["country", "total_confirmed"]])
print()
# large Asian countries (two conditions, each parenthesized)
asia_big = covid[(covid["continent"] == "Asia") & (covid["population"] > 100_000_000)]
print(asia_big[["country", "population"]])

     country  total_confirmed
26    Brazil         30682094
72    France         29160802
78   Germany         25780226
94     India         43121599
212       UK         22159805
216      USA         84209473

         country  population
15    Bangladesh   167745162
44         China  1439323776
94         India  1405273033
95     Indonesia   278910317
103        Japan   125755488
153     Pakistan   228878790
159  Philippines   112297269


## Derived columns

Whole-column math builds new features: cases as a share of population, and a case-fatality ratio (deaths per 100 confirmed).

In [24]:
covid["cases_per_capita"] = covid["total_confirmed"] / covid["population"]
covid["cfr_pct"] = covid["total_deaths"] / covid["total_confirmed"] * 100

print(covid[["country", "cases_per_capita", "cfr_pct"]].head())

       country  cases_per_capita   cfr_pct
0  Afghanistan          0.004420  4.289691
1      Albania          0.095954  1.268988
2      Algeria          0.005865  2.586376
3      Andorra          0.543983  0.362938
4       Angola          0.002853  1.915438


## Missing data

Several columns have gaps. Count them, then decide per column whether to drop or fill.

In [25]:
print(covid.isna().sum())          # missing values per column
print()

subset = covid[["country", "total_deaths", "total_tests"]]
print("rows total     :", len(subset))
print("rows w/o any NaN:", len(subset.dropna()))
print()

# a modeling choice: treat unreported tests as 0
covid["total_tests"] = covid["total_tests"].fillna(0)
print("tests still missing:", covid["total_tests"].isna().sum())

country                            0
continent                          0
total_confirmed                    0
total_deaths                       8
total_recovered                   22
active_cases                      22
serious_or_critical               81
total_cases_per_1m_population      0
total_deaths_per_1m_population     8
total_tests                       14
total_tests_per_1m_population     14
population                         0
cases_per_capita                   0
cfr_pct                            8
dtype: int64

rows total     : 226
rows w/o any NaN: 210

tests still missing: 0


## Group and summarize by continent

In [26]:
by_cont = covid.groupby("continent")[["total_confirmed", "total_deaths"]].sum()
print(by_cont.sort_values("total_confirmed", ascending=False))
print()
print(covid["continent"].value_counts())

                   total_confirmed  total_deaths
continent                                       
Europe                   194330079     1830655.0
Asia                     149999659     1427939.0
North America             99625662     1467234.0
South America             57136485     1296523.0
Africa                    12042400      254319.0
Australia/Oceania          7942867       11413.0

continent
Africa               58
Asia                 49
Europe               48
North America        39
Australia/Oceania    18
South America        14
Name: count, dtype: int64


## Build: a small pipeline

Among countries with a sizeable population (> 5M), find the highest reported deaths per million, then total confirmed cases by continent for that filtered set.

In [27]:
step = covid[covid["population"] > 5_000_000].copy()               # 1. filter
step = step.dropna(subset=["total_deaths_per_1m_population"])       # 2. clean
step = step.sort_values("total_deaths_per_1m_population",
                        ascending=False)                           # 3. sort
print(step[["country", "continent", "total_deaths_per_1m_population"]].head())
print()
print(step.groupby("continent")["total_confirmed"].sum()
          .sort_values(ascending=False))                           # 4. group

            country      continent  total_deaths_per_1m_population
158            Peru  South America                          6297.0
29         Bulgaria         Europe                          5407.0
92          Hungary         Europe                          4820.0
55   Czech Republic         Europe                          3745.0
184        Slovakia         Europe                          3667.0

continent
Europe               187232788
Asia                 144992013
North America         97815196
South America         56007223
Africa                10694182
Australia/Oceania      7681354
Name: total_confirmed, dtype: int64


## Your turn

**Micro-assignment.** Eight problems on NumPy and Pandas; see `../micro-assignment/README.md`.

**Next, class 1.5 (APIs and the web):** your first LLM call, local and hosted.